# 02 - Image Analysis Basics

This notebook builds on `01_python_basics.ipynb`. By the end you will be able to:

1. Load, display and inspect images (shape, data type, intensities)
2. Crop, flip and split colour channels
3. Handle image data types and adjust contrast
4. Filter images to reduce noise and find edges
5. Segment objects with thresholding
6. Clean masks with morphology
7. Label objects and measure them in a table
8. Save your results

**How to use this notebook:** run each cell with `Shift + Enter`, in order from top to bottom. Cells starting with **Exercise** are for you to try.

## 0. Imports

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import ndimage as ndi
from skimage import data, exposure, filters, io, morphology, segmentation, util
from skimage.color import label2rgb, rgb2gray
from skimage.measure import label, regionprops_table
from skimage.util import img_as_float, img_as_ubyte

plt.rcParams['figure.dpi'] = 100

---
## 1. Loading and displaying images

scikit-image ships with example images, so no download is needed. To load your own file use `io.imread('path/to/file.tif')`.

In [ ]:
coins = data.coins()
print('type :', type(coins))
print('shape:', coins.shape)
print('dtype:', coins.dtype)
print('min / max:', coins.min(), coins.max())

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(coins, cmap='gray')
ax.set_title('Coins')
fig.colorbar(im, ax=ax, label='intensity')
plt.show()

### 1.1 Colormaps
A grayscale image has no colour of its own. The colormap (`cmap`) decides how numbers are drawn. `vmin` and `vmax` set the display range without changing the data.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, cmap in zip(axes, ['gray', 'viridis', 'magma', 'gray']):
    ax.imshow(coins, cmap=cmap)
    ax.set_title(cmap)
    ax.axis('off')
axes[3].imshow(coins, cmap='gray', vmin=50, vmax=150)
axes[3].set_title('gray, vmin=50, vmax=150')
plt.tight_layout()
plt.show()

### 1.2 Intensity histogram
The histogram shows how many pixels have each intensity. Two peaks usually mean background and objects.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(coins.ravel(), bins=256, range=(0, 256), color='0.3')
ax.set_xlabel('intensity')
ax.set_ylabel('pixel count')
ax.set_title('Histogram of the coins image')
plt.show()

---
## 2. Manipulating images

### 2.1 Cropping and flipping
An image is an array, so cropping is just slicing.

In [ ]:
crop = coins[50:150, 100:250]          # rows 50-149, columns 100-249
flipped_lr = coins[:, ::-1]            # reverse the columns
flipped_ud = np.flipud(coins)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, img, title in zip(axes, [crop, flipped_lr, flipped_ud], ['crop', 'flip left-right', 'flip up-down']):
    ax.imshow(img, cmap='gray')
    ax.set_title(f'{title} {img.shape}')
plt.tight_layout()
plt.show()

### 2.2 Colour images and channels
An RGB image has a third axis: `(rows, columns, channels)`.

In [ ]:
cat = data.chelsea()
print('shape:', cat.shape, ' dtype:', cat.dtype)

fig, axes = plt.subplots(1, 5, figsize=(18, 3.5))
axes[0].imshow(cat)
axes[0].set_title('RGB')
for i, (name, cmap) in enumerate([('red', 'Reds_r'), ('green', 'Greens_r'), ('blue', 'Blues_r')]):
    axes[i + 1].imshow(cat[:, :, i], cmap=cmap)
    axes[i + 1].set_title(f'{name} channel')
gray = rgb2gray(cat)
axes[4].imshow(gray, cmap='gray')
axes[4].set_title(f'grayscale ({gray.dtype})')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

### 2.3 Data types matter
| dtype | range |
|---|---|
| `uint8` | 0 to 255 |
| `uint16` | 0 to 65535 (common in microscopy) |
| `float64` | usually 0.0 to 1.0 in scikit-image |

Integer arithmetic can **overflow** silently. Convert to float before doing maths.

In [ ]:
pixel = np.array([200], dtype=np.uint8)
print('uint8 200 + 100 =', pixel + np.uint8(100), ' <- overflow! It wraps around past 255.')

coins_f = img_as_float(coins)
print('as float: dtype', coins_f.dtype, ' range', coins_f.min(), 'to', coins_f.max())
print('back to uint8:', img_as_ubyte(coins_f).dtype)

### 2.4 Contrast adjustment

In [ ]:
p2, p98 = np.percentile(coins, (2, 98))
stretched = exposure.rescale_intensity(coins, in_range=(p2, p98))
equalized = exposure.equalize_adapthist(coins)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, img, title in zip(axes, [coins, stretched, equalized], ['original', 'contrast stretch (2-98%)', 'adaptive equalisation (CLAHE)']):
    ax.imshow(img, cmap='gray')
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()
plt.show()

**Exercise 1:** Load `data.camera()`. Print its shape and dtype, crop out the photographer's head and display it with the `magma` colormap.

In [ ]:
# Your code here


---
## 3. Filtering

Filters replace each pixel with a value computed from its neighbours.
- **Gaussian**: smooths and reduces noise
- **Median**: removes salt-and-pepper noise while keeping edges
- **Sobel**: highlights edges

In [ ]:
noisy = util.random_noise(coins, mode='s&p', amount=0.05, rng=0)

gauss = filters.gaussian(noisy, sigma=2)
median = filters.median(noisy, morphology.disk(2))
edges = filters.sobel(coins_f)

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, img, title in zip(axes, [noisy, gauss, median, edges], ['noisy', 'Gaussian (sigma=2)', 'median (disk 2)', 'Sobel edges']):
    ax.imshow(img, cmap='gray')
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()
plt.show()

---
## 4. Segmentation by thresholding

Thresholding turns an image into a **binary mask**: `True` for object pixels, `False` for background. Otsu's method picks the threshold automatically from the histogram.

The coins image has uneven lighting, so a single global threshold struggles. A **local** threshold compares each pixel with its own neighbourhood.

In [ ]:
smooth = filters.gaussian(coins_f, sigma=1)

t_otsu = filters.threshold_otsu(smooth)
mask_global = smooth > t_otsu

t_local = filters.threshold_local(smooth, block_size=95, offset=-0.02)
mask_local = smooth > t_local

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(smooth.ravel(), bins=256, color='0.3')
axes[0].axvline(t_otsu, color='C3', label=f'Otsu = {t_otsu:.2f}')
axes[0].legend()
axes[0].set_title('histogram')
axes[1].imshow(mask_global, cmap='gray')
axes[1].set_title('global Otsu threshold')
axes[2].imshow(mask_local, cmap='gray')
axes[2].set_title('local threshold')
for ax in axes[1:]:
    ax.axis('off')
plt.tight_layout()
plt.show()

---
## 5. Cleaning masks with morphology

- `ndi.binary_fill_holes` fills holes inside objects
- `binary_opening` removes thin protrusions and small specks
- `remove_small_objects` drops objects below a size
- `clear_border` drops objects touching the image edge

In [ ]:
clean = ndi.binary_fill_holes(mask_local)
clean = morphology.binary_opening(clean, morphology.disk(3))
clean = morphology.remove_small_objects(clean, min_size=150)
clean = segmentation.clear_border(clean)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(mask_local, cmap='gray')
axes[0].set_title('raw mask')
axes[1].imshow(clean, cmap='gray')
axes[1].set_title('cleaned mask')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

---
## 6. Labelling and measuring objects

`label` gives every connected object its own integer ID. `regionprops_table` measures each object and returns a table that pandas can read.

In [ ]:
labels = label(clean)
print('Number of objects found:', labels.max())

overlay = label2rgb(labels, image=coins, bg_label=0, alpha=0.4)
fig, ax = plt.subplots(figsize=(7, 5))
ax.imshow(overlay)
ax.contour(clean, levels=[0.5], colors='yellow', linewidths=0.8)
ax.set_title(f'{labels.max()} labelled coins')
ax.axis('off')
plt.show()

In [ ]:
props = regionprops_table(
    labels,
    intensity_image=coins,
    properties=['label', 'area', 'centroid', 'eccentricity', 'intensity_mean', 'equivalent_diameter_area'],
)
df = pd.DataFrame(props)
df.head(10)

In [ ]:
print(df[['area', 'intensity_mean', 'eccentricity']].describe().round(2))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].hist(df['area'], bins=15, color='C0')
axes[0].set_xlabel('area (pixels)')
axes[0].set_ylabel('count')
axes[1].scatter(df['area'], df['intensity_mean'], color='C0')
axes[1].set_xlabel('area (pixels)')
axes[1].set_ylabel('mean intensity')
plt.tight_layout()
plt.show()

**Exercise 2:** Filter `df` to keep only coins with `area` above the median area. How many are there? *Hint:* `df[df['area'] > df['area'].median()]`

In [ ]:
# Your code here


---
## 7. Saving results

`pathlib.Path` builds file paths that work on Windows, macOS and Linux alike, so never hard-code `\` or `/` in paths.

In [ ]:
out_dir = Path('output')
out_dir.mkdir(exist_ok=True)

io.imsave(out_dir / 'coins_labels.tif', labels.astype(np.uint16), check_contrast=False)
io.imsave(out_dir / 'coins_overlay.png', img_as_ubyte(overlay), check_contrast=False)
df.to_csv(out_dir / 'coins_measurements.csv', index=False)

for f in sorted(out_dir.iterdir()):
    print(f, f'{f.stat().st_size / 1024:.1f} KB')

---
## 8. Putting it together

Here is the whole workflow as one reusable function.

In [ ]:
def segment_and_measure(image, sigma=1, block_size=95, offset=-0.02, min_size=150):
    """Segment bright objects in a grayscale image and return (labels, table)."""
    img = filters.gaussian(img_as_float(image), sigma=sigma)
    mask = img > filters.threshold_local(img, block_size=block_size, offset=offset)
    mask = ndi.binary_fill_holes(mask)
    mask = morphology.binary_opening(mask, morphology.disk(3))
    mask = morphology.remove_small_objects(mask, min_size=min_size)
    mask = segmentation.clear_border(mask)
    lab = label(mask)
    table = pd.DataFrame(regionprops_table(lab, intensity_image=image, properties=['label', 'area', 'intensity_mean']))
    return lab, table

lab, table = segment_and_measure(coins)
print(f"{len(table)} objects, mean area {table['area'].mean():.0f} px")

**Exercise 3:** Call `segment_and_measure` with different `min_size` and `block_size` values and watch how the object count changes. Which settings give one object per coin?

In [ ]:
# Your code here
